# LossGuard cost-sensitive model walkthrough

## tl;dr

The executed 100,000-row run achieved held-out ROC AUC 0.9943 and average precision 0.8862. Its cost search selected global verify/decline thresholds of 0.275/0.900 and a validation cost of $8,429.60. These are bounded-run simulation results, not final production or causal claims.

## Context & Methods

The decision objective is simulated dollar cost, not accuracy. `fraudTrain.csv` is ordered by event time into separate 70% training, 15% calibration, and 15% threshold-validation windows; `fraudTest.csv` is used only for final evaluation.

### Key Assumptions

Margin, LTV, verification efficacy, and reacquisition costs are documented in `../docs/business-assumptions.md`. The source is simulated and dashboard savings are retrospective estimates.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from ml.train import train

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "dataset" / "fraudTrain.csv"
TEST_DATA_PATH = PROJECT_ROOT / "dataset" / "fraudTest.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "fraud_model.joblib"
METADATA_PATH = PROJECT_ROOT / "models" / "model_metadata.json"
MAX_ROWS = 100_000  # Increase or set to None for the full training file.
assert DATA_PATH.exists(), f"Missing source: {DATA_PATH}"
assert TEST_DATA_PATH.exists(), f"Missing final test source: {TEST_DATA_PATH}"

## Data

Validate the source columns and show a bounded label summary without displaying direct customer identifiers.

In [ ]:
preview_columns = ["trans_date_trans_time", "category", "amt", "is_fraud"]
preview = pd.read_csv(DATA_PATH, usecols=preview_columns, nrows=MAX_ROWS)
pd.DataFrame(
    {
        "rows": [len(preview)],
        "fraud_rows": [int(preview["is_fraud"].sum())],
        "fraud_rate": [float(preview["is_fraud"].mean())],
        "categories": [int(preview["category"].nunique())],
        "min_event_time": [preview["trans_date_trans_time"].min()],
        "max_event_time": [preview["trans_date_trans_time"].max()],
    }
)

## Results

Train the shared implementation and persist the API-ready bundle and auditable JSON metadata.

In [ ]:
metadata = train(
    str(DATA_PATH),
    str(TEST_DATA_PATH),
    str(MODEL_PATH),
    str(METADATA_PATH),
    max_rows=MAX_ROWS,
)
summary_keys = [
    "model_version",
    "windows",
    "calibration",
    "final_test",
    "global_thresholds",
    "global_threshold_validation_cost",
]
{key: metadata[key] for key in summary_keys}

In [ ]:
thresholds = (
    pd.DataFrame(metadata["category_thresholds"])
    .T.reset_index(names="merchant_category")
    .sort_values("decline")
)
numeric_columns = ["verify", "decline", "validation_rows", "validation_fraud_rows"]
thresholds[numeric_columns] = thresholds[numeric_columns].apply(pd.to_numeric)
px.scatter(
    thresholds,
    x="verify",
    y="decline",
    color="merchant_category",
    size="validation_rows",
    hover_data=["validation_fraud_rows"],
    title="Cost-optimal validation thresholds by merchant category",
)

## Takeaways

The held-out ranking metrics show strong separation on this bounded time window, while the threshold chart shows why category economics can favor different actions. The 0.99% fraud rate in this early 100,000-row slice is higher than the full training file's 0.58%, so do not generalize these figures to the complete dataset. Before publishing portfolio claims, train on the intended scope, evaluate once on the untouched test file, and quote model version `xgb-20260811T121909Z` with the documented assumptions.